In [4]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' can be imported
import sys
from pathlib import Path # for path manipulations
# Move three levels up from current working directory to reach workspace root
workspace_root = Path.cwd().parent.parent.parent.resolve() 
smx_dir = workspace_root / 'smx'  # Path to smx folder
if str(smx_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(smx_dir)) # insert at the start of sys.path to prioritize local modules

#Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{workspace_root}/real_datasets/xrf/soyben.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1.40':'23.10']  # selecting only the spectral features (columns with numeric names)

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1.40':'23.10'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1.40':'23.10'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)


Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

2026-03-10 07:09:47,863 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-03-10 07:09:47,872 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-03-10 07:09:47,977 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-03-10 07:09:47,983 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.



In [5]:
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=2,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
#vip_scores_mat.T.plot()
plsda_results[0]

,LVs,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy CV,Sensitivity CV,Specificity CV,CM CV,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred,X Cum Exp Var,Y Cum Exp Var,X Ind Exp Var,Y Ind Exp Var
0,1,0.640625,0.178571,1.000000,"[[36, 0], [23, 5]]",0.1875,0.035714,0.305556,"[[11, 25], [27, 1]]",0.551724,0.000000,1.0000,"[[16, 0], [13, 0]]",35.285791,2.955985,35.285791,2.955985
1,2,0.750000,0.642857,0.833333,"[[30, 6], [10, 18]]",0.5625,0.428571,0.666667,"[[24, 12], [16, 12]]",0.758621,0.538462,0.9375,"[[15, 1], [6, 7]]",47.497340,20.981041,12.211548,18.025056


In [6]:
Xcalclass.T.plot()

## Definição das Zonas Espectrais

In [7]:
spectral_cuts = [
('Al', 1.40, 1.66),
('Si', 1.66, 1.86),
('P', 1.86, 2.16),
('S', 2.16, 2.44),
('Rh L + Ar', 2.44, 3.10),
('K', 3.10, 3.46),
('Ca ka', 3.46, 3.86),
('Ca kb', 3.86, 4.18),
('Ti ka', 4.18, 4.66),
('background1', 5.08, 5.72),
('Mn', 5.72, 6.14),
('Fe ka', 6.14, 6.68),
('Fe kb', 6.68, 7.20),
('Ni', 7.20, 7.70),
('Cu', 7.70, 8.30),
('Zn ka', 8.30, 9.10),
('Zn kb', 9.10, 9.87),
('background2', 9.87, 13.00),
('Rb', 13.00, 13.75),
('Sr', 13.75, 14.61),
('Zr', 14.61, 15.50),
('background3', 15.50, 17.70),
('Compton ka', 17.70, 19.80),
('Rayleigh ka', 19.80, 20.58),
('Compton kb', 20.58, 22.12),
('Rayleigh kb', 22.12, 23.10),
]

## VIP, Regression Coefficients e SHAP (como no original)

In [8]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

In [9]:
shap_global_importance = pd.read_csv('shap_soyben.csv', sep=';') # loading previously saved shap_unique_df

# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
shap_unique_df

,energy,Mean_Abs_SHAP,Zone
0,6.37,0.015023,Fe ka
1,8.61,0.014527,Zn ka
2,19.14,0.011377,Compton ka
3,3.38,0.009249,K
4,20.30,0.005949,Rayleigh ka
5,14.21,0.003696,Sr
6,3.80,0.003196,Ca ka
7,7.11,0.002052,Fe kb
8,13.31,0.001576,Rb
9,5.89,0.001527,Mn


In [10]:
# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False)
shap_unique_df = shap_unique_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df

,energy,Mean_Abs_SHAP,Zone
0,6.37,0.015023,Fe ka
1,8.61,0.014527,Zn ka
2,19.14,0.011377,Compton ka
3,3.38,0.009249,K
4,20.30,0.005949,Rayleigh ka
5,14.21,0.003696,Sr
6,3.80,0.003196,Ca ka
7,7.11,0.002052,Fe kb
8,13.31,0.001576,Rb
9,5.89,0.001527,Mn


## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [11]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = exp.aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zone 'Al': VE = 55.54%, dim = 14 variables
Zone 'Si': VE = 54.98%, dim = 11 variables
Zone 'P': VE = 51.00%, dim = 16 variables
Zone 'S': VE = 44.42%, dim = 15 variables
Zone 'Rh L + Ar': VE = 35.74%, dim = 34 variables
Zone 'K': VE = 75.72%, dim = 19 variables
Zone 'Ca ka': VE = 60.16%, dim = 21 variables
Zone 'Ca kb': VE = 42.52%, dim = 17 variables
Zone 'Ti ka': VE = 26.26%, dim = 25 variables
Zone 'background1': VE = 15.02%, dim = 32 variables
Zone 'Mn': VE = 28.44%, dim = 21 variables
Zone 'Fe ka': VE = 98.64%, dim = 27 variables
Zone 'Fe kb': VE = 92.97%, dim = 26 variables
Zone 'Ni': VE = 21.03%, dim = 25 variables
Zone 'Cu': VE = 20.61%, dim = 30 variables
Zone 'Zn ka': VE = 52.61%, dim = 40 variables
Zone 'Zn kb': VE = 21.01%, dim = 39 variables
Zone 'background2': VE = 51.35%, dim = 157 variables
Zone 'Rb': VE = 72.91%, dim = 38 variables
Zone 'Sr': VE = 75.87%, dim = 44 variables
Zone 'Zr': VE = 78.49%, dim = 45 variables
Zone 'background3': VE = 81.72%, dim = 110 variables


In [12]:
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred_cont
)

In [13]:
# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_pred_conticted_numeric = pd.Series(y_pred_cont) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred_conticted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        perturbation_mode='median',
        stats_source='full',
        aim='regression',
        metric='mean_abs_diff',
        normalize_by_zone_size=True,   # ← ativa a normalização
        zone_size_exponent=1.0,        # ← √n (moderado); use 1.0 para n completo
        verbose=True
    )

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graph(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Samples: Yes | Predicates: No (All) | Valid: 161 | Discarded: 47
Bag_2 | Samples: Yes | Predicates: No (All) | Valid: 178 | Discarded: 30
Bag_3 | Samples: Yes | Predicates: No (All) | Valid: 156 | Discarded: 52
Bag_4 | Samples: Yes | Predicates: No (All) | Valid: 169 | Discarded: 39
Bag_5 | Samples: Yes | Predicates: No (All) | Valid: 178 | Discarded: 30
Bag_6 | Samples: Yes | Predicates: No (All) | Valid: 163 | Discarded: 45
Bag_7 | Samples: Yes | Predicates: No (All) | Valid: 162 | Discarded: 46
Bag_8 | Samples: Yes | Predicates: No (All) | Valid: 167 | Discarded: 41
Bag_9 | Samples: Yes | Predicates: No (All) | Valid: 175 | Discarded: 33
Bag_10 | Samples: Yes | Predicates: No (All) | Valid: 162 | Discarded: 46
PERTURBATION IMPORTANCE FOR PREDICATES
Task type (aim): regression
Perturbation mode: median
Statistics source: full
Metric: mean_abs_diff
Total folds: 10
Zone-size normalization: ENABLED (exponent=1.0)


[Bag_1] Processing 161 predicates...
  

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Rayleigh ka > 0.17,K > 0.10,Fe ka > 0.20,Fe ka > 0.20
1,Zn ka > 0.09,Fe ka > 0.20,K <= -0.35,Rayleigh ka <= -0.45
2,Fe ka > 0.20,Rayleigh ka > -0.08,Zn ka <= -0.06,Fe ka > -0.51
3,Rayleigh ka > -0.08,Rayleigh ka > 0.61,Zn ka > 0.24,Zn ka > 0.09
4,Rayleigh ka > 0.61,Rayleigh ka <= -0.45,Rayleigh ka <= -0.45,Rayleigh ka > 0.61
...,...,...,...,...
200,Ti ka > 0.09,Ni <= 0.05,NaN,Class_A
201,background3 > 0.69,Ni > -0.04,NaN,Class_B
202,background1 <= -0.08,Ni <= 0.00,NaN,NaN
203,Class_A,Class_A,NaN,NaN


In [14]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,K > 0.30,0.002187
1,Zn ka > 0.09,0.002054
2,Rayleigh ka > 0.17,0.002046
3,Zn ka <= -0.06,0.001946
4,Fe ka > -0.51,0.001859
...,...,...
156,background3 > 0.21,0.000091
157,background3 <= 0.21,0.000090
158,background3 > -0.60,0.000083
159,background3 <= 0.69,0.000083


In [15]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Fe ka > 0.20,4.496036,Fe ka,0.20,>
1,Rayleigh ka > 0.61,3.968191,Rayleigh ka,0.61,>
2,Zn ka > 0.24,3.864212,Zn ka,0.24,>
3,K > 0.10,3.523537,K,0.10,>
4,Ca ka > 0.23,3.140880,Ca ka,0.23,>
5,Compton kb <= -0.56,2.570937,Compton kb,-0.56,<=
6,Sr <= -0.34,2.445429,Sr,-0.34,<=
7,Mn > 0.12,2.245269,Mn,0.12,>
8,Compton ka > 1.10,2.112159,Compton ka,1.10,>
9,Zr <= -0.35,1.198956,Zr,-0.35,<=


In [16]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = exp.aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_pert_natural

Zone 'Al': VE = 55.89%, dim = 14 variables
Zone 'Si': VE = 55.01%, dim = 11 variables
Zone 'P': VE = 51.19%, dim = 16 variables
Zone 'S': VE = 44.29%, dim = 15 variables
Zone 'Rh L + Ar': VE = 33.87%, dim = 34 variables
Zone 'K': VE = 86.68%, dim = 19 variables
Zone 'Ca ka': VE = 75.18%, dim = 21 variables
Zone 'Ca kb': VE = 50.79%, dim = 17 variables
Zone 'Ti ka': VE = 26.72%, dim = 25 variables
Zone 'background1': VE = 15.25%, dim = 32 variables
Zone 'Mn': VE = 39.81%, dim = 21 variables
Zone 'Fe ka': VE = 99.45%, dim = 27 variables
Zone 'Fe kb': VE = 96.24%, dim = 26 variables
Zone 'Ni': VE = 14.60%, dim = 25 variables
Zone 'Cu': VE = 29.19%, dim = 30 variables
Zone 'Zn ka': VE = 71.13%, dim = 40 variables
Zone 'Zn kb': VE = 23.10%, dim = 39 variables
Zone 'background2': VE = 56.54%, dim = 157 variables
Zone 'Rb': VE = 73.07%, dim = 38 variables
Zone 'Sr': VE = 76.07%, dim = 44 variables
Zone 'Zr': VE = 78.51%, dim = 45 variables
Zone 'background3': VE = 81.75%, dim = 110 variables


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Fe ka > 0.20,4.496036,Fe ka,0.20,>,0.114600,9.0,0.037587,Fe ka > 0.114600
1,Rayleigh ka > 0.61,3.968191,Rayleigh ka,0.61,>,0.928593,31.0,0.005251,Rayleigh ka > 0.928593
2,Zn ka > 0.24,3.864212,Zn ka,0.24,>,0.157867,21.0,0.007783,Zn ka > 0.157867
3,Rayleigh ka <= -0.45,3.812978,Rayleigh ka,-0.45,<=,-0.654311,43.0,0.018916,Rayleigh ka <= -0.654311
4,Zn ka > 0.09,3.774683,Zn ka,0.09,>,0.067920,41.0,0.011108,Zn ka > 0.067920
...,...,...,...,...,...,...,...,...,...
202,Ni <= 0.00,0.389582,Ni,0.00,<=,0.006415,42.0,0.005439,Ni <= 0.006415
203,Rayleigh kb <= -0.35,0.358344,Rayleigh kb,-0.35,<=,-0.358182,42.0,0.000043,Rayleigh kb <= -0.358182
204,P <= -0.12,0.326598,P,-0.12,<=,-0.067320,19.0,0.000425,P <= -0.067320
205,Class_A,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone'] # o n indica a linha do dataframe lrc_summed_df_pert_natural que queremos acessar para pegar o nome da zona espectral (coluna 'Zone')
threshold_score = float(lrc_summed_df_pert_natural.iloc[n]['Threshold_Natural']) # o iloc acessa a linha n e a coluna 'Threshold_Natural' para pegar o valor do threshold já mapeado para o espaço natural (não pré-processado)

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
#print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Fe ka':
  - Dimensão: 27 variáveis espectrais
  - Variância explicada pela PC1: 99.45%


In [18]:
# Permutation importance baseado em mudança nas predições do modelo PLS
# Medimos a média da diferença absoluta entre as predições originais e as predições
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Predições base do modelo PLS
baseline_pred = pls_model.predict(Xcalclass_prep)
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_pred = pls_model.predict(X_perm)
        diffs.append(np.mean(np.abs(baseline_pred - perm_pred)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance': importance_list
})
permutation_df.sort_values(by='Permutation_importance', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance', ascending=False)
permutation_unique_df

,energy,Permutation_importance,Zone
0,6.37,0.017083,Fe ka
1,19.14,0.014691,Compton ka
2,8.65,0.012119,Zn ka
3,20.38,0.011483,Rayleigh ka
4,3.38,0.008879,K
5,14.21,0.007897,Sr
6,21.50,0.006601,Compton kb
7,3.78,0.005907,Ca ka
8,7.11,0.005667,Fe kb
9,5.89,0.005519,Mn


In [19]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'VIP_Score': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_Coefficient' : pad_list(reg_vet_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,VIP_Score,Reg_Coefficient,Shap,Permutation,LRC_perturbation
0,Fe ka,Zn ka,Fe ka,Fe ka,Fe ka
1,Zn ka,Mn,Zn ka,Compton ka,Rayleigh ka
2,Fe kb,Rayleigh ka,Compton ka,Zn ka,Zn ka
3,Rayleigh ka,Ca ka,K,Rayleigh ka,K
4,Mn,Sr,Rayleigh ka,K,Ca ka
5,Ca ka,Rb,Sr,Sr,Compton kb
6,Sr,Compton ka,Ca ka,Compton kb,Sr
7,Rb,background2,Fe kb,Ca ka,Mn
8,Compton ka,K,Rb,Fe kb,Compton ka
9,background2,Compton kb,Mn,Mn,Zr


In [20]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

,Method_1,Method_2,RBO_Score
7,Shap,Permutation,0.829765
1,VIP_Score,Shap,0.797887
9,Permutation,LRC_perturbation,0.749552
8,Shap,LRC_perturbation,0.743331
3,VIP_Score,LRC_perturbation,0.722673
2,VIP_Score,Permutation,0.707349
0,VIP_Score,Reg_Coefficient,0.349158
4,Reg_Coefficient,Shap,0.297535
6,Reg_Coefficient,LRC_perturbation,0.276624
5,Reg_Coefficient,Permutation,0.211297


In [21]:
lrc_summed_df_pert_natural.to_csv('lrc_pert_natural.csv', index=False, sep=';')